# Project 1 · Multilayer Perceptron on MNIST
**AI Workshop – From MLP to Transformers**

Teach a network to read handwritten digits. We will:
1. **Load MNIST**: 70,000 handwritten digits, 28 × 28 pixels each.
2. **Build the MLP**: 784 → 128 → 64 → 10.
3. **Train** it with loss and gradient descent.
4. **Evaluate** it on test images it has never seen.

> **Before you start:** *Runtime → Change runtime type → T4 GPU*, then run the cells from top to bottom (**Shift + Enter**).

## 0 · Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)

## 01 · Load MNIST
`ToTensor()` turns each image into numbers and scales the pixels from 0–255 down to 0–1, which keeps training stable.

In [ ]:
transform = transforms.ToTensor()   # pixels 0–255 → 0–1, shape 1 × 28 × 28

train_set = datasets.MNIST("data", train=True, download=True, transform=transform)
test_set = datasets.MNIST("data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=256)

print("Training images:", len(train_set))
print("Test images:    ", len(test_set))

To a computer, an image is just a grid of numbers. Let's look at one.

In [ ]:
image, label = train_set[0]
print("Shape:", tuple(image.shape), "  label:", label)

# the middle of the digit, as brightness values 0–255
grid = (image[0, 4:24, 4:24] * 255).int()
for row in grid:
    print(" ".join(f"{v:3d}" for v in row.tolist()))

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    img, lab = train_set[i]
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(lab)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 02 · Build the MLP
`Flatten` turns the 28 × 28 image into a row of 784 numbers. Two hidden layers with **ReLU**, then 10 outputs, one score per digit.

We don't add a softmax here: `CrossEntropyLoss` applies it for us during training.

> ✏️ **Your turn:** replace every `TODO` in the cell below, then run it. Stuck? The solution is in the **Solutions** section at the end of the notebook.

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                    # 1 × 28 × 28 → 784
            # ✏️ Fill in: 784 → 128 → 64 → 10, with ReLU after each hidden layer
            nn.Linear(784, TODO), nn.ReLU(),  # hidden layer 1: 128 neurons
            nn.Linear(TODO, TODO), TODO,        # hidden layer 2: 64 neurons, then ReLU
            nn.Linear(TODO, TODO),             # output: 10 scores (logits)
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"Learnable parameters: {n_params:,}", "✅" if n_params == 109_386 else "❌ expected 109,386")

## 03 · Train
Every batch runs the same four steps you saw in the animation:

| Step | Code |
|---|---|
| 1 · Forward pass | `logits = model(images)` |
| 2 · Loss | `loss = loss_fn(logits, labels)` |
| 3 · Backpropagation | `loss.backward()` |
| 4 · Update: w ← w − η · ∂L/∂w | `optimizer.step()` |

We use plain **SGD** (gradient descent) with learning rate **η = 0.1**.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)   # η = 0.1

> ✏️ **Your turn:** replace every `TODO` in the cell below, then run it. Stuck? The solution is in the **Solutions** section at the end of the notebook.

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        # ✏️ Fill in: the four steps of training
        logits = TODO                        # 1 · forward pass: run the model on the images
        loss = TODO                          # 2 · loss: compare the logits with the labels (use loss_fn)
        optimizer.zero_grad()               #     clear the gradients of the previous batch
        TODO                                 # 3 · backpropagation: compute the gradients of the loss
        TODO                                 # 4 · update the weights with the optimizer
        loss_sum += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return loss_sum / total, correct / total

Helpers to measure accuracy (ready to run).

In [ ]:
@torch.no_grad()
def evaluate(model, loader, shift=0):
    """Accuracy on a dataset. shift > 0 moves every image that many pixels to the right."""
    model.eval()
    total, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if shift:
            images = torch.roll(images, shifts=shift, dims=3)
        correct += (model(images).argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return correct / total

def count_params(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
EPOCHS = 5
history = {"loss": [], "train_acc": [], "test_acc": []}

for epoch in range(1, EPOCHS + 1):
    loss, train_acc = train_one_epoch(model, train_loader, optimizer)
    test_acc = evaluate(model, test_loader)
    history["loss"].append(loss)
    history["train_acc"].append(train_acc)
    history["test_acc"].append(test_acc)
    print(f"Epoch {epoch}: loss {loss:.3f} · train acc {train_acc:.1%} · test acc {test_acc:.1%}")

In [ ]:
epochs = range(1, len(history["loss"]) + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(epochs, history["loss"], marker="o")
ax1.set_title("Training loss"); ax1.set_xlabel("epoch")
ax2.plot(epochs, history["train_acc"], marker="o", label="train")
ax2.plot(epochs, history["test_acc"], marker="o", label="test")
ax2.set_title("Accuracy"); ax2.set_xlabel("epoch"); ax2.legend()
plt.tight_layout()
plt.show()

## 04 · Evaluate on test data
The test images were never used for training, so this is the honest score.

In [ ]:
print(f"Test accuracy: {evaluate(model, test_loader):.2%}")

images, labels = next(iter(test_loader))
with torch.no_grad():
    probs = F.softmax(model(images.to(device)), dim=1).cpu()   # softmax: scores → probabilities

fig, axes = plt.subplots(2, 8, figsize=(12, 3.8))
for i, ax in enumerate(axes.flat):
    pred = probs[i].argmax().item()
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(f"{pred} ({probs[i, pred]:.0%})", color="green" if pred == labels[i] else "red")
    ax.axis("off")
plt.suptitle("Prediction (confidence): green = correct, red = wrong")
plt.tight_layout()
plt.show()

Which digits does it get wrong? Some of them are hard for people too.

In [ ]:
wrong_images, wrong_preds, wrong_labels = [], [], []
with torch.no_grad():
    for images, labels in test_loader:
        preds = model(images.to(device)).argmax(dim=1).cpu()
        mask = preds != labels
        wrong_images += list(images[mask]); wrong_preds += preds[mask].tolist(); wrong_labels += labels[mask].tolist()

print("Mistakes:", len(wrong_images), "out of", len(test_set))
fig, axes = plt.subplots(2, 8, figsize=(12, 3.8))
for ax, img, p, t in zip(axes.flat, wrong_images, wrong_preds, wrong_labels):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"said {p}, was {t}", color="red", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
torch.save(model.state_dict(), "mlp_mnist.pt")
print("Saved mlp_mnist.pt")

## Bonus · Try your own digit with OpenCV
Write a digit with a dark pen on white paper, take a photo, and upload it. OpenCV turns it into an MNIST-style image:
**read → grayscale → invert → crop → resize → normalize**.

MNIST digits are white on black, about 20 × 20 pixels, centred in a 28 × 28 frame, so we do the same.

In [ ]:
import cv2
import numpy as np

def to_mnist(path):
    img = cv2.imread(path)                              # OpenCV loads colour images as BGR
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)        # 3 channels → 1 channel
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    # dark ink on white paper → white digit on black, like MNIST
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    ys, xs = np.nonzero(bw)
    digit = bw[ys.min():ys.max() + 1, xs.min():xs.max() + 1]   # crop to the digit
    h, w = digit.shape
    scale = 20 / max(h, w)
    digit = cv2.resize(digit, (max(1, round(w * scale)), max(1, round(h * scale))), interpolation=cv2.INTER_AREA)
    canvas = np.zeros((28, 28), dtype=np.uint8)
    y0, x0 = (28 - digit.shape[0]) // 2, (28 - digit.shape[1]) // 2
    canvas[y0:y0 + digit.shape[0], x0:x0 + digit.shape[1]] = digit
    return canvas / 255.0                                # 0–255 → 0–1

try:
    from google.colab import files
    uploaded = files.upload()                            # choose a photo of your digit
    path = next(iter(uploaded))
    x = to_mnist(path)
    with torch.no_grad():
        p = F.softmax(model(torch.tensor(x, dtype=torch.float32).view(1, 1, 28, 28).to(device)), dim=1)[0].cpu()
    plt.figure(figsize=(7, 3))
    plt.subplot(1, 2, 1); plt.imshow(x, cmap="gray"); plt.axis("off"); plt.title("What the model sees")
    plt.subplot(1, 2, 2); plt.bar(range(10), p); plt.xticks(range(10)); plt.title(f"Prediction: {p.argmax().item()}")
    plt.tight_layout(); plt.show()
except ImportError:
    print("This cell uses Colab's upload button. Outside Colab, call to_mnist('your_image.png') instead.")

## 🔑 Solutions
Only if you are stuck: run the solution for your step (double-click to see the code), then go back and continue where you were.

In [ ]:
# @title Solution · Build the MLP
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),                    # 1 × 28 × 28 → 784
            nn.Linear(784, 128), nn.ReLU(),  # hidden layer 1
            nn.Linear(128, 64), nn.ReLU(),   # hidden layer 2
            nn.Linear(64, 10),               # output: 10 scores (logits)
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"Learnable parameters: {n_params:,}", "✅" if n_params == 109_386 else "❌ expected 109,386")

In [ ]:
# @title Solution · train_one_epoch
def train_one_epoch(model, loader, optimizer):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)              # 1 · forward pass
        loss = loss_fn(logits, labels)      # 2 · loss
        optimizer.zero_grad()
        loss.backward()                     # 3 · backpropagation
        optimizer.step()                    # 4 · update the weights
        loss_sum += loss.item() * len(labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return loss_sum / total, correct / total